In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from scipy.signal import correlate, correlation_lags

In [ ]:

d = (X_vars['Food (y/y%)']-X_vars['Food (y/y%)'].mean())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), dpi=80)

plot_acf(d.tolist(), ax=ax1, lags=50)
plot_pacf(d.tolist(), ax=ax2, lags=20)

# Decorate the plots
ax1.spines["top"].set_alpha(.3); ax2.spines["top"].set_alpha(.3)
ax1.spines["bottom"].set_alpha(.3); ax2.spines["bottom"].set_alpha(.3)
ax1.spines["right"].set_alpha(.3); ax2.spines["right"].set_alpha(.3)
ax1.spines["left"].set_alpha(.3); ax2.spines["left"].set_alpha(.3)
ax1.set_xticks(np.arange(0,52,2))
ax2.set_xticks(np.arange(0,22,2))
# ax1.tick_params(axis='both', labelsize=12)
# ax2.tick_params(axis='both', labelsize=12)

plt.show()

In [ ]:
d = (X_vars['Food (y/y%)']-X_vars['Food (y/y%)'].mean())

max_ar = 4
max_ma = 4

# Initialize a list to store results
results_list = []

# Loop over all combinations of p and q
for p in range(max_ar + 1):
    for q in range(max_ma + 1):
        try:
            # Create and fit the SARIMAX model (ARMA(p, q) with no differencing)
            model = SARIMAX(d, order=(p, 0, q), trend='c')  # 'c' for constant term
            results = model.fit(disp=False)  # disp=False to suppress convergence messages
            
            # Append the results: p, q, BIC
            results_list.append((p, q, results.bic))
            
            # Optional: print progress
#             print(f"ARMA({p},{q}) - BIC: {results.bic:.4f}")
            
        except Exception as e:
            # If model fails to converge or is invalid, append None
#             print(f"ARMA({p},{q}) - Failed: {e}")
            results_list.append((p, q, None))

# Convert results to a DataFrame
results_df = pd.DataFrame(results_list, columns=['p', 'q', 'BIC'])

# Filter out failed models
results_df = results_df.dropna(subset=['BIC'])

# Sort by BIC in ascending order to find the best model
best_model = results_df.loc[results_df['BIC'].idxmin()]

print(f"\nBest ARMA model based on BIC: ARMA({best_model['p']},{best_model['q']}) with BIC = {best_model['BIC']:.4f}")



In [ ]:
# Fit an ARIMA model
# The order (p, d, q) should be determined based on ACF/PACF plots or using auto_arima
#d, differencing, p PCAF cut off, q ACF cut off
model = ARIMA(np.array((X_vars['Food (y/y%)']-X_vars['Food (y/y%)'].mean()).diff(1)), order=(1, 0, 3))  # Example order
fitted_model = model.fit()

# Obtain the residuals
residuals = fitted_model.resid
x_white = residuals.copy()
# Check if residuals are white noise
# Plot residuals

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 3))

# ax1.figure(figsize=(10, 4))
ax1.plot(residuals)
ax1.set_title('Residuals from ARIMA Model')
# plt.show()

# Plot ACF of residuals to check for autocorrelation
lags = 40
acf_values = acf(residuals, nlags=lags)
# ax2.figure(figsize=(10, 4))
ax2.stem(acf_values)
ax2.set_title('ACF of Residuals')
plt.show()

In [ ]:
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from scipy.signal import correlate

y_white = np.squeeze(y_filtered) #np.squeeze(np.array(y_values.copy()))
y_white = y_white[~np.isnan(y_white)]
x_white =  x_white[~np.isnan(x_white)] #important to remove NANs
# x_white.dropna(inplace=True)
min_len = min(len(x_white), len(y_white))
x_white = x_white[:min_len]
y_white = y_white[:min_len]

ccf = correlate(x_white, y_white)#, mode='full')
lags = np.arange(-min_len+1, min_len)
lags = np.arange(-len(x_white)+1, len(x_white))
plt.figure(figsize=(20, 6))
plt.stem(lags, ccf)
plt.title('Cross-Correlation between Whitened Data and Target Variable')
plt.xlabel('Lag')
plt.ylabel('Cross-Correlation')
plt.xlim(0, len(x_white))
plt.xticks(np.arange(0,150, 2), size = 8)
plt.grid(True)
plt.show()

#highest stem shows lagg


In [ ]:
#make lags
dataframe = pd.DataFrame()
for i in range(12,0,-1):
    dataframe['t-'+str(i)] = X_vars['Food (y/y%)'].shift(i)
dataframe['t'] = np.array(X_vars['Food (y/y%)'])
# print(dataframe.head(13))
dataframe = dataframe[13:]
dataframe

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from matplotlib import pyplot

array = dataframe.values
# split into input and output
X = array[:,0:-1]
y = array[:,-1]
# fit random forest model
model = RandomForestRegressor(n_estimators=500, random_state=1)
model.fit(X, y)
# show importance scores
print(model.feature_importances_)
# plot importance scores
names = dataframe.columns.values[0:-1]
ticks = [i for i in range(len(names))]
pyplot.bar(ticks, model.feature_importances_)
pyplot.title('Feature importances for lagged features')
pyplot.xticks(ticks, names)
pyplot.show()

In [ ]:
Compare with unlagged

def timeseries_feature_selection(data, y, difference, max_ar, max_ma, shifts, plot = False): #Box - Jenkins approach
    #centralise data
    col_name = pd.DataFrame(data).columns[0]    
    data = (data - data.mean())
    
    if plot is True: #plot ACF and PCAF
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4), dpi=80)
        plot_acf(data.tolist(), ax=ax1, lags=50)
        plot_pacf(data.tolist(), ax=ax2, lags=20)
        ax1.set_title(label = 'Autocorrelation', size = 20)
        ax2.set_title(label ='Partial Autocorrelation', size = 20)

        # Decorate the plots
        ax1.spines["top"].set_alpha(.3); ax2.spines["top"].set_alpha(.3)
        ax1.spines["bottom"].set_alpha(.3); ax2.spines["bottom"].set_alpha(.3)
        ax1.spines["right"].set_alpha(.3); ax2.spines["right"].set_alpha(.3)
        ax1.spines["left"].set_alpha(.3); ax2.spines["left"].set_alpha(.3)
        ax1.set_xticks(np.arange(0,52,2))
        ax2.set_xticks(np.arange(0,22,2))
        fig.suptitle(f'Feature selection for {col_name}', size = 22)
        fig.subplots_adjust(top=0.75)
        plt.show()
    
    #initialise results list
    results_list = []
    
    for p in range(max_ar + 1):
        for q in range(max_ma + 1):
            try:
                # Create and fit the SARIMAX model (ARMA(p, q) with no differencing)
                model = SARIMAX(data, order=(p, 0, q), trend='c')  # 'c' for constant term
                results = model.fit(disp=False)  # disp=False to suppress convergence messages

                # Append the results: p, q, BIC
                results_list.append((p, q, results.bic))

                # Optional: print progress
    #             print(f"ARMA({p},{q}) - BIC: {results.bic:.4f}")

            except Exception as e:
                # If model fails to converge or is invalid, append None
                print(f"ARMA({p},{q}) - Failed: {e}")
                results_list.append((p, q, None))
    
    results_df = pd.DataFrame(results_list, columns=['p', 'q', 'BIC'])

    # Filter out failed models
    results_df = results_df.dropna(subset=['BIC'])

    # Sort by BIC in ascending order to find the best model
    best_model = results_df.loc[results_df['BIC'].idxmin()]
    
    print(f"\nBest ARMA model based on BIC: ARMA({best_model['p']},{best_model['q']}) with BIC = {best_model['BIC']:.4f}")
    
    p = best_model['p']
    q = best_model['q']
    #================================ARIMA==============================
    #Fit ARIMA to p, q
    model = ARIMA(np.array(data.diff(difference)), order=(p, 0, q))  # Example order
    fitted_model = model.fit()

    # Obtain the residuals
    residuals = fitted_model.resid
    x_white = residuals.copy()
    
    if plot is True: #plot ARIMA residuals
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 3))
        ax1.plot(residuals)
        ax1.set_title('Residuals from ARIMA Model', size = 20)

        # Plot ACF of residuals to check for autocorrelation
        lags = 40
        acf_values = acf(residuals, nlags=lags)
        ax2.stem(acf_values)
        ax2.set_title('ACF of Residuals', size = 20)
        plt.show()    
    
    #apply filter to y data
    ar_params = fitted_model.params
    y_filtered = convolution_filter(y, np.r_[1, -ar_params], nsides=1)
    
    y_white = np.squeeze(y_filtered) #np.squeeze(np.array(y_values.copy()))
    y_white = y_white[~np.isnan(y_white)]
    x_white =  x_white[~np.isnan(x_white)] #important to remove NANs
    # x_white.dropna(inplace=True)
    min_len = min(len(x_white), len(y_white))
    x_white = x_white[:min_len]
    y_white = y_white[:min_len]

    ccf = correlate(x_white, y_white)#, mode='full')
    lags = np.arange(-min_len+1, min_len)
    lags = np.arange(-len(x_white)+1, len(x_white))
    ##================================Plot correllogram================================
    if plot is True:
        plt.figure(figsize=(20, 6))
        plt.stem(lags, ccf)
        plt.title('Cross-Correlation between Whitened Data and Target Variable', size = 20)
        plt.xlabel('Lag')
        plt.ylabel('Cross-Correlation')
        plt.xlim(0, len(x_white))
        plt.xticks(np.arange(0,150, 2), size = 8)
        plt.grid(True)
        plt.show()
    
    #get lags
    dataframe = pd.DataFrame()
    for i in range(shifts, 0 ,-1):  #range(shifts) to include unlagged
        dataframe['t-'+str(i+1)] = data.diff(difference).shift(i)
    dataframe['t'] = np.array(X_vars['Food (y/y%)'].diff(difference))
#     print(dataframe.head(13))
    dataframe = dataframe[shifts:]  #shifts+1 to include unlagged
    
    ##================================RFE================================
    array = dataframe.values
    # split into input and output
    X = array[:,0:-1] #remove -1 to include unlagged
    y = array[:,-1]
    # fit random forest model
    model = RandomForestRegressor(n_estimators=500, random_state=1)
    model.fit(X, y)
    # show importance scores
    print(model.feature_importances_)
    # plot importance scores
    names = dataframe.columns.values[0:-1]  #remove -1 to include unlagged
    ticks = [i for i in range(len(names))]
    
    
    #=====================================================================
    #return the best variable column as df
    fimp = pd.DataFrame({'Feature':names, 'importance':model.feature_importances_})
#     fimp.nlargest(1, 'importance')
    fimp = fimp.nlargest(1, 'importance')
    res = dataframe[fimp['Feature']]
    name = col_name + ' ' + res.columns[0]
    res.rename(columns={res.columns[0]: name}, inplace=True)
    #================================Plot selected feature================================
    if plot is True:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 3))

        ax1.bar(ticks, model.feature_importances_)
        ax1.set_title('Feature importances for lagged features', size = 20)
        ax1.set_xticks(ticks, names)
    #     pyplot.show()

        ax2.plot(res, label = f'{name}', color = '#bf9b30')
        ax2.plot(y_values.diff(difference), label = f'{y_values.columns[0]}', color = 'teal')   
        ax2.set_title(f'Selected {name} vs {y_values.columns[0]}', size = 20)
        ax2.legend()
        plt.show()
    
    return #res #np.as_array(fimp.nlargest(1, 'importance'))
    
    

In [ ]:
# #now engineer the features
X_selected = pd.concat([y_values, X_selected], axis = 1) #add y vlaues in for feature engineering

X_engineered = engineer_features(X_selected, y_values)

# #create a bunch of features, then remove features with low variance
# #get rolling average, srolling sum, slope coefficients for 1 year
# X_roll_mean = roll_mean_features(X_selected, 4)
# X_roll_mean = variance_selector(X_roll_mean, 3)

# #rolling sum
# X_roll_sum = roll_sum_features(X_selected, 4)
# X_roll_sum = variance_selector(X_roll_sum, 10)

# #rolling standard deviation
# X_roll_sd = roll_sd_features(X_selected, 4)
# X_roll_sd = variance_selector(X_roll_sd, 0.1)

# #expanding mean, sum and std
# X_expand = expand_features(X_selected)
# X_expand = variance_selector(X_expand, 0.2)

# #slope coeffs
# X_slopes = slope_coeffs_features(X_selected, 4)
# X_slopes = variance_selector(X_slopes, 0.3)

# # #%percent difference
# X_change = percent_change_features(X_selected, 4)
# X_change = variance_selector(X_change, 50)
# X_change.replace([-np.inf], np.nanmin(X_change[X_change != -np.inf]), inplace=True)
# X_change.replace([np.inf], np.nanmax(X_change[X_change != np.inf]), inplace=True)
# X_change_white = pd.DataFrame(whiten(X_change), columns = X_change.columns).add_suffix('_whitened').set_index(y_values.index)

# # #fourier series
# X_fourier = fourier_features(X_selected)
# X_fourier = X_fourier.set_index(y_values.index)
# # X_fourier = variance_selector(X_fourier, 0.01)

# #prewhiten all the variables except change and fourier. This function returns the most important pre-whitened features in each df.
# #use 4 ma and 4 ar because 4 quarters in a year. difference by 1 to make staitonary the variables. 
# lagged_vars = create_whiten_lag(X_selected, y_values, 1, 4, 4, 4)
# lagged_roll_mean = create_whiten_lag(X_roll_mean, y_values, 1, 4, 4, 4)
# lagged_roll_sum = create_whiten_lag(X_roll_sum, y_values, 1, 4, 4, 4)
# lagged_roll_sd = create_whiten_lag(X_roll_sd, y_values, 1, 4, 4, 4)
# lagged_expand = create_whiten_lag(X_expand, y_values, 1, 4, 4, 4)
# lagged_slopes = create_whiten_lag(X_slopes, y_values, 1, 4, 4, 4)

# # #all features 458 columns
# X_engineered = pd.concat([X_selected, lagged_vars, X_roll_mean, lagged_roll_mean, X_roll_sum, lagged_roll_sum, 
#                           X_roll_sd, lagged_roll_sd, X_expand, lagged_expand, X_slopes, lagged_slopes, 
#                           X_change, X_change_white, X_fourier], axis = 1)
# X_engineered = X_engineered.fillna(0) 

In [ ]:
  def _timeseries_predict(X, y, periods):
        dates = y.index
        y_Trend, y_Season, y_Resid = seasonal_decompose_df(y, 'additive', periods)
        
        preds = pred = pd.DataFrame((np.array(y_Season)[:,0] + np.array(X) + np.array(y_Trend)[:,0]), columns = ['prediction']).set_index(y.index)
        return preds

In [ ]:
def _xg_huber(y, y_pred, delta):
    scale = 1 + ((y-y_pred)/ delta)**2 
    scale_sqrt = np.sqrt(scale)
    gradient = (y - y_pred) / scale_sqrt
    hessian = 1.0 / scale / scale_sqrt
    return gradient, hessian

_xg_huber(y_train, residuals, 0.2)

In [ ]:
def huber_loss(residuals, epsilon): #instead of MSE
    abs_resid = np.abs(residuals)
    quadratic = 0.5 * residuals**2
    linear = epsilon * (abs_resid - 0.5 * epsilon)
    
    return np.where(abs_resid <= epsilon, quadratic, linear).sum()

def huber_gradient(residuals, epsilon): #as sign vector in MAE
    abs_resid = np.abs(residuals)
    gradient = np.where(abs_resid <= epsilon, -residuals, -epsilon * np.sign(residuals))
    return gradient

def huber_initialise_predictions(y): #to start the preds
    func = lambda i: huber_loss(y, i)
    initials = y.mean()
    result = minimize(fun = func, x0 = initials)
    
    return result.x[0]

def initialize_predictions(y, loss_function): 
    if loss_function is 'huber': return huber_initialise_predictions(y)
    else: return y.mean()
    
def huber(residuals, left_mask, right_mask, epsilon, y):
    huber_residuals = huber_gradient(residuals, epsilon)
     
    left_value = np.mean(huber_residuals[left_mask]) if np.any(left_mask) else 0
    right_value = np.mean(huber_residuals[right_mask]) if np.any(right_mask) else 0
    
#     print("LEFT PRED : RIGHT PRED: ", left_value, right_value)
    
    y_pred = np.full_like(huber_residuals, left_value)
    y_pred[right_mask] = right_value
#     print(y_pred)
#     print("LEFT, RIGHT: ", left_value, right_value)
        
    error = huber_loss(compute_residuals(huber_residuals, y_pred), epsilon)

    return error, left_value, right_value


In [ ]:
def compute_residuals(y, y_pred): return y - y_pred
def initialize_predictions(y): return y.mean()

def mse_(residuals, left_mask, right_mask):
    left_value = residuals[left_mask].mean()
    right_value = residuals[right_mask].mean()
    
    mse = (np.mean((residuals[left_mask] - left_value) ** 2) * np.sum(left_mask) + 
       np.mean((residuals[right_mask] - right_value) ** 2) * np.sum(right_mask)) / len(residuals)
    return mse, left_value, right_value

def abs_error(residuals, left_mask, right_mask): #note, residuals have to be median, not mean for abs_error: outliers
    sign_vector = np.sign(residuals)
    
    left_value = sign_vector[left_mask].mean()
    right_value = sign_vector[right_mask].mean()
    
    mae = (np.mean((sign_vector[left_mask] - left_value) ** 2) * np.sum(left_mask) +
           np.mean((sign_vector[right_mask] - right_value) ** 2) * np.sum(right_mask)) / len(residuals)
    
    return mae, left_value, right_value

def friedman_MSE(residuals, left_mask, right_mask, weights = None):
    if weights is None:
        weights = np.ones(len(residuals))
   
    left_value = residuals[left_mask].mean()
    right_value = residuals[right_mask].mean()

    w_l = np.sum(weights[left_mask]) 
    w_r = np.sum(weights[right_mask])
    
    mean_l = np.mean(residuals[left_mask])
    mean_r = np.mean(residuals[right_mask])
    
    friedman_mse = ((w_l * w_r) / (w_l + w_r)) * ((mean_l - mean_r)**2)

    return friedman_mse, left_value, right_value

In [ ]:
def correlate_xicor(df):
    batch = df.corr(method = xicor)
    return batch1['Consumers price index (CPI) (q/q%)'].sort_values(ascending=False)